In [1]:
from resources.imports import *

import torch
import torch.nn as nn

from resources.MLdata import DATA
from resources.MLfunc import EarlyStopping, CombinedCurveLoss, QuantileLoss, QuantileLossMATLAB
from resources.MLfunc import hOpt_model, hOpt_compare, hOpt_best_summary
from resources.MLmodels import *

In [2]:
%load_ext autoreload
%autoreload 2

# Stress-Strain Curve

## Multi-Layer Perceptrion (MLP)

### DATA

In [3]:
DAT_MLP = DATA(
    path=1, 
    path_add="",
    load=True, 
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC", 
    dis="disNodes", 
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="MLP",
    scale=("symm", "inout"),
    reduce_dim=False, #("PCA", "out", 0.95, 10, True)
    round_decimals=5
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
DAT = DAT_MLP
if DAT.UTmechTest:
    print("UT shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("Train -  in (min, max):", DAT.UT_train_in.min(), DAT.UT_train_in.max(), "\n        out (min, max):", DAT.UT_train_out.min(), DAT.UT_train_out.max())
    print("  Val -  in (min, max):", DAT.UT_val_in.min(), DAT.UT_val_in.max(), "\n        out (min, max):", DAT.UT_val_out.min(), DAT.UT_val_out.max())
    print(" Test -  in (min, max):", DAT.UT_test_in.min(), DAT.UT_test_in.max(), "\n        out (min, max):", DAT.UT_test_out.min(), DAT.UT_test_out.max())
    if DAT.FTmechTest: print("\n========================================================\n")
if DAT.FTmechTest:
    print("FT shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("Train -  in (min, max):", DAT.FT_train_in.min(), DAT.FT_train_in.max(), "\n        out (min, max):", DAT.FT_train_out.min(), DAT.FT_train_out.max())
    print("  Val -  in (min, max):", DAT.FT_val_in.min(), DAT.FT_val_in.max(), "\n        out (min, max):", DAT.FT_val_out.min(), DAT.FT_val_out.max())
    print(" Test -  in (min, max):", DAT.FT_test_in.min(), DAT.FT_test_in.max(), "\n        out (min, max):", DAT.FT_test_out.min(), DAT.FT_test_out.max())

UT shape: (5561, 1444) (5561, 201)
Train -  in (min, max): -1.0 1.0 
        out (min, max): -1.0 1.0
  Val -  in (min, max): -1.0 1.0 
        out (min, max): -1.6191947669483455 1.103937947512592
 Test -  in (min, max): -1.0 1.0 
        out (min, max): -1.595077081066206 1.0862485225493743


### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Full HPO run on the full DATA split. Re-running resumes the saved Optuna study.

MLP_hOpt_model_space = {
    "depth": [2, 3, 4, 5],
    "width": [128, 256, 512],
    "block": ["mlp", "res"],
    "act": ["relu", "gelu", "mish"],
    "norm": [None, "batch", "layer"],
    "dropout": {"type": "float", "low": 0.0, "high": 0.25},
    "head_norm": [None, "layer"],
    "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
}

MLP_hOpt_loss_space = {
    "family": ["mse", "combined"],
    "mse_weight": [0.25, 0.5, 1.0],
    "weighted_mse_weight": [0.0, 0.5, 1.0, 2.0],
    "derivative_weight": [0.0, 0.02, 0.05, 0.10],
    "peak_weight": [0.0, 0.10, 0.25, 0.50],
    "energy_weight": [0.0, 0.05, 0.10, 0.25],
    "peak_location_weight": [0.0, 0.02, 0.05],
    "reduction": ["mean"],
    "derivative_order": [1],
    "SoftPeak_beta": [10.0, 20.0, 40.0],
    "UT": {
        "zone_boundaries": (67, 134),
        "zone_weights": (1.0, 5.0, 2.0),
    },
}

MLP_hOpt_train_space = {
    "optimizer": ["adamw"],
    "lr": {"type": "float", "low": 5e-5, "high": 2e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-8, "high": 1e-3, "log": True},
    "batch": [16, 32, 64],
    "n_epochs": {"type": "fixed", "value": 250},
    "metric": ["rmse"],
    "scheduler": ["plateau"],
    "scheduler_factor": [0.3, 0.5],
    "scheduler_patience": [10, 20],
    "scheduler_threshold": {"type": "fixed", "value": 1e-4},
    "early_stop": [True],
    "early_stop_patience": [30, 50],
    "early_stop_min_delta": {"type": "fixed", "value": 1e-5},
    "verbose": {"type": "fixed", "value": 0},
}

study_MLP_full = hOpt_model(
    typ="mlp",
    data=DAT_MLP,
    n_trials=50,
    model_space=MLP_hOpt_model_space,
    loss_space=MLP_hOpt_loss_space,
    train_space=MLP_hOpt_train_space,
    seed=42,
    device=device,
    save=True,
    save_best_model=True,
    name="MLP_full_hOpt",
    n_jobs=1,
    show_progress_bar=True,
)

hOpt_best_summary(study_MLP_full)

c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2200: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-14 18:59:34,323] A new study created in RDB with name: MLP_full_hOpt


  0%|          | 0/50 [00:00<?, ?it/s]

 -> Epoch 1/250 || LOSS - train: 53.583874, val: 14.438961 | MSE - train: 0.410989, val: 0.381455 | RMSE - train: 0.641084, val: 0.617620 | LR: 1.58e-04
================ Training Complete ================
 Best Epoch: 78, with LOSS: 0.945959, MSE: 0.138678 and RMSE: 0.372395 ==================
UT val metrics | MAE: 3.241986, MSE: 24.011948, RMSE: 4.900199, mean summed abs err: 651.639166
Best prediction: 342, Worst prediction: 326
UT VAL prediction diagnostics | collapse ratio: 0.0578 | RMSE: 4.9 | mean-curve RMSE: 4.633 | skill vs mean curve: -0.0577 | peak corr: 0.0138 | energy corr: 0.0892 | sample curve corr: 0.985
UT VAL zones | elastic: collapse 0.145, RMSE 1.172 | peak_region: collapse 0.0434, RMSE 7.552 | post_peak: collapse 0.0775, RMSE 3.692
[I 2026-05-14 19:02:54,729] Trial 0 finished with value: 0.37239496613984546 and parameters: {'mlp_act': 'gelu', 'mlp_norm': None, 'mlp_dropout': 0.014520903042049865, 'mlp_head_dropout': 0.17323522915498704, 'mlp_depth': 5, 'mlp_width': 

### MODEL

In [ ]:
MLP1 = MODEL(
    typ=DAT.model,
    model=MLP(in_size=DAT.UT_train_in.shape[-1], 
              h_size=[4096, 2048, 1024, 1024, 1024, 512, 512], 
              out_size=DAT.UT_train_out.shape[-1], 
              act="relu",
              block="mlp",
              norm="batch", 
              dropout=0.0).to(device), 
    lossf=QuantileLossMATLAB(quantiles=[0.5, 0.45, 0.1], zone_boundaries=(67, 134), err_type="L2"), #nn.MSELoss(reduction="mean"), #CustomQuantileLoss(quantiles=[0.5, 0.45, 0.1], zone_boundaries=(50, 150), err_type="L2"),
    opt=("adam", 6.016e-6), #("adam", 1e-4),
    batch=32,
    lr=9e-4,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.2427, 100, 1e-4), #("min", 0.2427, 18, 1e-4), 
    earlyStop=EarlyStopping(patience=500, min_delta=1e-4, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True
)

MLP1.summary()

In [ ]:
MLP1.train(n_epochs=1000, verbose=20, plot=True)    

In [ ]:
MLP1.save(path=None, name=None)

In [ ]:
MLP1.predict(test_dataloader=None, plot=True)

In [ ]:
for i in range(len(MLP1.UT_test_outputs)):
    plot_predictions(DAT.UT_OUT_df, MLP1.UT_test_outputs, MLP1.UT_truth, indx=i, d_out=False)

## Graph Neural Network (GNN)

### DATA

In [ ]:
DAT_GNN = DATA(
    path=1, 
    path_add="",
    load=True, 
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC", 
    dis="disNodes", 
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="GNN",
    scale=("symm", "inout"),
    reduce_dim=False, #("PCA", "out", 0.95, 10, True)
    round_decimals=5,
    geom_feats=(True, True),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
DAT = DAT_GNN
if DAT.UTmechTest:
    print("UT shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("Train -  in (min, max):", DAT.UT_train_in.min(), DAT.UT_train_in.max(), "\n        out (min, max):", DAT.UT_train_out.min(), DAT.UT_train_out.max())
    print("  Val -  in (min, max):", DAT.UT_val_in.min(), DAT.UT_val_in.max(), "\n        out (min, max):", DAT.UT_val_out.min(), DAT.UT_val_out.max())
    print(" Test -  in (min, max):", DAT.UT_test_in.min(), DAT.UT_test_in.max(), "\n        out (min, max):", DAT.UT_test_out.min(), DAT.UT_test_out.max())
    if DAT.FTmechTest: print("\n========================================================\n")
if DAT.FTmechTest:
    print("FT shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("Train -  in (min, max):", DAT.FT_train_in.min(), DAT.FT_train_in.max(), "\n        out (min, max):", DAT.FT_train_out.min(), DAT.FT_train_out.max())
    print("  Val -  in (min, max):", DAT.FT_val_in.min(), DAT.FT_val_in.max(), "\n        out (min, max):", DAT.FT_val_out.min(), DAT.FT_val_out.max())
    print(" Test -  in (min, max):", DAT.FT_test_in.min(), DAT.FT_test_in.max(), "\n        out (min, max):", DAT.FT_test_out.min(), DAT.FT_test_out.max())

UT shape: (5561, 800, 8) (5561, 201)
Train -  in (min, max): -1.0 1.0 
        out (min, max): -1.0 1.0
  Val -  in (min, max): -1.0 1.0 
        out (min, max): -1.6191947669483455 1.103937947512592
 Test -  in (min, max): -1.0 1.0 
        out (min, max): -1.595077081066206 1.0862485225493743


### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Full GCN/GAT HPO run on the full graph DATA split. Re-running resumes the saved Optuna studies.

GNN_hOpt_model_space = {
    "gcn": {
        "depth": [2, 3, 4],
        "width": [64, 128, 256],
        "act": ["relu", "gelu", "mish"],
        "norm": [None, "layer"],
        "dropout": {"type": "float", "low": 0.0, "high": 0.25},
        "head_norm": [None, "layer"],
        "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
        "pool": ["mean"],
    },
    "gat": {
        "depth": [1, 2, 3],
        "width": [32, 64, 128],
        "heads": [1, 2, 4],
        "act": ["relu", "gelu"],
        "norm": [None, "layer"],
        "dropout": {"type": "float", "low": 0.0, "high": 0.25},
        "att_dropout": {"type": "float", "low": 0.0, "high": 0.20},
        "head_norm": [None, "layer"],
        "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
        "pool": ["mean"],
    },
}

GNN_hOpt_loss_space = {
    "family": ["mse", "combined"],
    "mse_weight": [0.25, 0.5, 1.0],
    "weighted_mse_weight": [0.0, 0.5, 1.0, 2.0],
    "derivative_weight": [0.0, 0.02, 0.05, 0.10],
    "peak_weight": [0.0, 0.10, 0.25, 0.50],
    "energy_weight": [0.0, 0.05, 0.10, 0.25],
    "peak_location_weight": [0.0, 0.02, 0.05],
    "reduction": ["mean"],
    "derivative_order": [1],
    "SoftPeak_beta": [10.0, 20.0, 40.0],
    "UT": {
        "zone_boundaries": (67, 134),
        "zone_weights": (1.0, 5.0, 2.0),
    },
}

GNN_hOpt_train_space = {
    "optimizer": ["adamw"],
    "lr": {"type": "float", "low": 3e-5, "high": 1e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-8, "high": 1e-3, "log": True},
    "batch": [4, 8, 16],
    "n_epochs": {"type": "fixed", "value": 150},
    "metric": ["rmse"],
    "scheduler": ["plateau"],
    "scheduler_factor": [0.3, 0.5],
    "scheduler_patience": [8, 15],
    "scheduler_threshold": {"type": "fixed", "value": 1e-4},
    "early_stop": [True],
    "early_stop_patience": [25, 40],
    "early_stop_min_delta": {"type": "fixed", "value": 1e-5},
    "verbose": {"type": "fixed", "value": 0},
}

studies_GNN_full = hOpt_compare(
    typs=["gcn", "gat"],
    data=DAT_GNN,
    n_trials_per_typ=30,
    model_space=GNN_hOpt_model_space,
    loss_space=GNN_hOpt_loss_space,
    train_space=GNN_hOpt_train_space,
    seed=42,
    device=device,
    save=True,
    save_best_model=True,
    name="GNN_full_hOpt",
    n_jobs=1,
    show_progress_bar=True,
)

hOpt_best_summary(studies_GNN_full)

c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2188: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-13 22:02:47,415] A new study created in RDB with name: GNN_full_hOpt_gcn


  0%|          | 0/30 [00:00<?, ?it/s]

 -> Epoch 1/150 || LOSS - train: 0.118420, val: 0.108052 | MSE - train: 0.101341, val: 0.091219 | RMSE - train: 0.318342, val: 0.302025 | LR: 7.28e-04
================ Training Complete ================
 Best Epoch: 28, with LOSS: 0.104125, MSE: 0.086588 and RMSE: 0.294258 ==================
UT val metrics | MAE: 2.766400, MSE: 21.543513, RMSE: 4.641499, mean summed abs err: 556.046488
Best prediction: 394, Worst prediction: 472
UT VAL prediction diagnostics | collapse ratio: 0.00828 | RMSE: 4.641 | mean-curve RMSE: 4.633 | skill vs mean curve: -0.0019 | peak corr: 0.0451 | energy corr: -0.0168 | sample curve corr: 0.986
UT VAL zones | elastic: collapse 0.0197, RMSE 1.152 | peak_region: collapse 0.0072, RMSE 7.469 | post_peak: collapse 0.00843, RMSE 2.742
[I 2026-05-13 22:26:01,636] Trial 0 finished with value: 0.2942576104976758 and parameters: {'gcn_act': 'gelu', 'gcn_norm': None, 'gcn_dropout': 0.03899863008405066, 'gcn_head_dropout': 0.011616722433639893, 'gcn_depth': 2, 'gcn_width

c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2188: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-14 17:51:01,439] A new study created in RDB with name: GNN_full_hOpt_gat


  0%|          | 0/30 [00:00<?, ?it/s]

 -> Epoch 1/150 || LOSS - train: 0.105807, val: 0.086612 | MSE - train: 0.105807, val: 0.086612 | RMSE - train: 0.325279, val: 0.294299 | LR: 2.56e-04
[W 2026-05-14 18:56:31,241] Trial 0 failed with parameters: {'gat_act': 'gelu', 'gat_norm': None, 'gat_dropout': 0.03900466011060913, 'gat_head_dropout': 0.031198904067240532, 'gat_depth': 2, 'gat_width': 128, 'gat_att_dropout': 0.16648852816008436, 'gat_head_norm': None, 'gat_heads': 4, 'gat_pool': 'mean', 'loss_family': 'mse', 'optimizer': 'adamw', 'lr': 0.00025638878443818403, 'weight_decay': 4.982752357076433e-08, 'batch': 16, 'objective_metric': 'rmse', 'w_init': 'auto', 'scheduler': 'plateau', 'scheduler_factor': 0.3, 'scheduler_patience': 15, 'early_stop': True, 'early_stop_patience': 40} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\exy053\Documents\00-PhD-gitRepo\.venv311\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(tri

KeyboardInterrupt: 

### MODEL

In [ ]:
GNN1 = MODEL(
    typ=DAT.model,
    model=GNN(in_size=DAT.UT_train_in.shape[-1], 
              h_size=[64, 64, 64, 64],
              out_size=DAT.UT_train_out.shape[-1], 
              act="relu",
              block="gat",
              norm="layer",
              dropout=0.2,
              bias=True,
              heads=2,
              pool="mean").to(device),
    lossf=nn.MSELoss(reduction="mean"),
    opt=("adam", 6.016e-6), #("adam", 1e-4),
    batch=36,
    lr=9e-4,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.2427, 100, 1e-4), 
    earlyStop=EarlyStopping(patience=150, min_delta=1e-4, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True
)

GNN1.summary()

In [ ]:
GNN1.train(n_epochs=1000, verbose=20, plot=True)

In [ ]:
GNN1.save(path=None, name=None)

In [ ]:
GNN1.predict(test_dataloader=None, plot=True)

In [ ]:
for i in range(len(GNN1.UT_test_outputs)):
    plot_predictions(DAT.UT_OUT_df, GNN1.UT_test_outputs, GNN1.UT_truth, indx=i, d_out=False)

## Transformer

### DATA

In [ ]:
DAT_TR = DATA(
    path=1, 
    path_add="",
    load=True, 
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC", 
    dis="disNodes", 
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="TR",
    scale=("symm", "inout"),
    reduce_dim=False, #("PCA", "out", 0.95, 10, True)
    round_decimals=5,
    geom_feats=(True, True)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
DAT = DAT_TR
if DAT.UTmechTest:
    print("UT shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("Train -  in (min, max):", DAT.UT_train_in.min(), DAT.UT_train_in.max(), "\n        out (min, max):", DAT.UT_train_out.min(), DAT.UT_train_out.max())
    print("  Val -  in (min, max):", DAT.UT_val_in.min(), DAT.UT_val_in.max(), "\n        out (min, max):", DAT.UT_val_out.min(), DAT.UT_val_out.max())
    print(" Test -  in (min, max):", DAT.UT_test_in.min(), DAT.UT_test_in.max(), "\n        out (min, max):", DAT.UT_test_out.min(), DAT.UT_test_out.max())
    if DAT.FTmechTest: print("\n========================================================\n")
if DAT.FTmechTest:
    print("FT shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("Train -  in (min, max):", DAT.FT_train_in.min(), DAT.FT_train_in.max(), "\n        out (min, max):", DAT.FT_train_out.min(), DAT.FT_train_out.max())
    print("  Val -  in (min, max):", DAT.FT_val_in.min(), DAT.FT_val_in.max(), "\n        out (min, max):", DAT.FT_val_out.min(), DAT.FT_val_out.max())
    print(" Test -  in (min, max):", DAT.FT_test_in.min(), DAT.FT_test_in.max(), "\n        out (min, max):", DAT.FT_test_out.min(), DAT.FT_test_out.max())

UT shape: (5561, 800, 8) (5561, 201)
Train -  in (min, max): -1.0 1.0 
        out (min, max): -1.0 1.0
  Val -  in (min, max): -1.0 1.0 
        out (min, max): -1.6191947669483455 1.103937947512592
 Test -  in (min, max): -1.0 1.0 
        out (min, max): -1.595077081066206 1.0862485225493743


### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Full HPO run on the full Transformer DATA split. Re-running resumes the saved Optuna study.

TR_hOpt_model_space = {
    "d_model": [32, 64, 128],
    "n_heads": [1, 2, 4, 8],
    "n_layers": [1, 2, 3],
    "ff_mult": [2, 4],
    "head_depth": [0, 1, 2],
    "head_width": [64, 128, 256],
    "pool": ["mean", "cls"],
    "use_cls_token": [True, False],
    "act": ["relu", "gelu"],
    "encoder_act": ["gelu", "relu"],
    "block": ["mlp", "res"],
    "norm": ["layer"],
    "dropout": {"type": "float", "low": 0.0, "high": 0.25},
    "att_dropout": {"type": "float", "low": 0.0, "high": 0.20},
    "head_norm": ["same", None, "layer"],
    "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
    "pos_encoding": ["learned", "sinusoidal"],
}

TR_hOpt_loss_space = {
    "family": ["mse", "combined"],
    "mse_weight": [0.25, 0.5, 1.0],
    "weighted_mse_weight": [0.0, 0.5, 1.0, 2.0],
    "derivative_weight": [0.0, 0.02, 0.05, 0.10],
    "peak_weight": [0.0, 0.10, 0.25, 0.50],
    "energy_weight": [0.0, 0.05, 0.10, 0.25],
    "peak_location_weight": [0.0, 0.02, 0.05],
    "reduction": ["mean"],
    "derivative_order": [1],
    "SoftPeak_beta": [10.0, 20.0, 40.0],
    "UT": {
        "zone_boundaries": (67, 134),
        "zone_weights": (1.0, 5.0, 2.0),
    },
}

TR_hOpt_train_space = {
    "optimizer": ["adamw"],
    "lr": {"type": "float", "low": 3e-5, "high": 1e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-8, "high": 1e-3, "log": True},
    "batch": [4, 8, 16],
    "n_epochs": {"type": "fixed", "value": 150},
    "metric": ["rmse"],
    "scheduler": ["plateau"],
    "scheduler_factor": [0.3, 0.5],
    "scheduler_patience": [8, 15],
    "scheduler_threshold": {"type": "fixed", "value": 1e-4},
    "early_stop": [True],
    "early_stop_patience": [25, 40],
    "early_stop_min_delta": {"type": "fixed", "value": 1e-5},
    "verbose": {"type": "fixed", "value": 0},
}

study_TR_full = hOpt_model(
    typ="tr",
    data=DAT_TR,
    n_trials=35,
    model_space=TR_hOpt_model_space,
    loss_space=TR_hOpt_loss_space,
    train_space=TR_hOpt_train_space,
    seed=42,
    device=device,
    save=True,
    save_best_model=True,
    name="TR_full_hOpt",
    n_jobs=1,
    show_progress_bar=True,
)

hOpt_best_summary(study_TR_full)

c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2269: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-06 22:02:17,151] A new study created in RDB with name: TR_full_hOpt


  0%|          | 0/35 [00:00<?, ?it/s]

c:\Programs\Python39\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


 -> Epoch 1/150 || LOSS - train: 0.100586, val: 0.087133 | MSE - train: 0.100586, val: 0.087133 | RMSE - train: 0.317153, val: 0.295183 | LR: 3.30e-04
================ Training Complete ================
 Best Epoch: 47, with LOSS: 0.086385, MSE: 0.086385 and RMSE: 0.293913 ==================
[I 2026-05-07 02:05:06,101] Trial 0 finished with value: 0.2939125272185462 and parameters: {'tr_act': 'gelu', 'tr_norm': 'layer', 'tr_dropout': 0.18299848545285127, 'tr_head_dropout': 0.11973169683940732, 'tr_d_model': 32, 'tr_n_heads': 1, 'tr_pool': 'mean', 'tr_use_cls_token': True, 'tr_head_depth': 2, 'tr_head_width': 256, 'tr_n_layers': 3, 'tr_ff_mult': 4, 'tr_encoder_act': 'relu', 'tr_head_block': 'mlp', 'tr_att_dropout': 0.12150897038028768, 'tr_head_norm': 'layer', 'tr_pos_encoding': 'learned', 'loss_family': 'mse', 'optimizer': 'adamw', 'lr': 0.00033046478547966453, 'weight_decay': 1.5876781526923984e-06, 'batch': 8, 'objective_metric': 'rmse', 'w_init': 'auto', 'scheduler': 'plateau', 'sch

c:\Programs\Python39\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


 -> Epoch 1/150 || LOSS - train: 0.127361, val: 0.087175 | MSE - train: 0.127361, val: 0.087175 | RMSE - train: 0.356876, val: 0.295255 | LR: 2.75e-04
================ Training Complete ================
 Best Epoch: 56, with LOSS: 0.086394, MSE: 0.086394 and RMSE: 0.293929 ==================
[I 2026-05-07 13:08:05,202] Trial 2 finished with value: 0.29392902224254286 and parameters: {'tr_act': 'gelu', 'tr_norm': 'layer', 'tr_dropout': 0.10677694715656408, 'tr_head_dropout': 0.16360295318449863, 'tr_d_model': 32, 'tr_n_heads': 1, 'tr_pool': 'mean', 'tr_use_cls_token': False, 'tr_head_depth': 1, 'tr_head_width': 128, 'tr_n_layers': 3, 'tr_ff_mult': 2, 'tr_encoder_act': 'relu', 'tr_head_block': 'mlp', 'tr_att_dropout': 0.09789055205551261, 'tr_head_norm': 'same', 'tr_pos_encoding': 'learned', 'loss_family': 'mse', 'optimizer': 'adamw', 'lr': 0.00027545227564861415, 'weight_decay': 1.471121537912197e-05, 'batch': 16, 'objective_metric': 'rmse', 'w_init': 'auto', 'scheduler': 'plateau', 'sc

### MODEL

In [ ]:
TR1 = MODEL(
    typ=DAT.model,
    model=Transformer(
        in_size=DAT.UT_train_in.shape[-1] if DAT.UTmechTest else DAT.FT_train_in.shape[-1],
        seq_len=DAT.UT_train_in.shape[-2] if DAT.UTmechTest else DAT.FT_train_in.shape[-2],
        h_size=[64],
        out_size=DAT.UT_train_out.shape[-1] if DAT.UTmechTest else DAT.FT_train_out.shape[-1],
        d_model=32,
        n_heads=4,
        n_layers=5,
        act="gelu",
        block="mlp",
        norm="layer",
        dropout=0.1,
        pool="mean"
    ).to(device),
    lossf={
    "UT":CombinedCurveLoss(
        mse_weight=1.0,
        weighted_mse_weight=3.0,
        derivative_weight=0.5,
        peak_weight=3.0,
        energy_weight=1.0,
        peak_location_weight=2.0,
        zone_boundaries=(50, 125),
        zone_weights=(1.0, 5.0, 2.0),
        x_values=DAT.UT_OUT_df if DAT.UTmechTest else None,
        reduction="mean",
        derivative_order=1,
        SoftPeak_beta=20.0,)}, 
    # "FT":CombinedCurveLoss(
    #     mse_weight=0.1,
    #     weighted_mse_weight=1.0,
    #     derivative_weight=0.5,
    #     peak_weight=0.2,
    #     energy_weight=0.2,
    #     peak_location_weight=0.05,
    #     zone_boundaries=(90, 150),
    #     zone_weights=(1.0, 5.0, 2.0),
    #     x_values=DAT.FT_OUT_df if DAT.FTmechTest else None,
    #     reduction="mean",
    #     derivative_order=1,
    #     SoftPeak_beta=20.0)
    # },
    opt=("adam", 6.016e-6), #("adam", 1e-4),
    batch=32,
    lr=1e-3,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.2427, 15, 1e-4), #("min", 0.2427, 18, 1e-4), 
    earlyStop=EarlyStopping(patience=18, min_delta=1e-4, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True
)

TR1.summary()

In [ ]:
TR1.train(n_epochs=10, verbose=1, plot=True)

In [ ]:
TR1.save(path=None, name=None)

In [ ]:
TR1.predict(test_dataloader=None, plot=True)

In [ ]:
for i in range(len(TR1.UT_test_outputs)):
   plot_predictions(DAT.UT_OUT_df, TR1.UT_test_outputs, TR1.UT_truth, indx=i, d_out=False)

# END

# Cleanup Ideas To Preserve

## 1. Uncertainty-Aware Curve Prediction

Goal: predict not only the mean stress-strain curve, but also uncertainty bands or calibrated confidence estimates.

Possible routes:
- Use model ensembles or MC dropout for practical uncertainty estimates.
- Add probabilistic output heads, e.g. mean + log-variance per curve point, trained with Gaussian negative log likelihood.
- Explore mixture-density outputs if curves are genuinely multimodal.
- Use GPR/BoTorch-style uncertainty only for low-dimensional latent or optimization tasks, since full stress-strain curves are likely too high-dimensional for direct GPR.

Why useful:
- Could identify unreliable predictions.
- Could support active learning or simulation selection.
- Could make optimization less overconfident.

## 2. Physics / Constraint Loss Extensions

Goal: encode simple mechanical expectations into training loss without forcing a full PINN formulation.

Current framework already includes curve-aware losses: weighted curve MSE, derivative loss, peak stress, strain energy, and soft peak location. Future extensions should build on those rather than resurrect the old PINN notebooks.

Possible constraints:
- Penalize negative stress where physically invalid.
- Encourage monotonic early elastic response.
- Penalize unrealistic oscillations or sharp nonphysical jumps.
- Add stiffness/slope consistency in the initial strain region.
- Add energy consistency using area under the curve.
- Add fracture/peak-region constraints separately for UT and FT.

Why useful:
- Keeps predictions mechanically plausible.
- May improve extrapolation.
- More compatible with current supervised framework than full DeepXDE/PINN code.

Related idea:
- Deep Ritz methods minimize an energy or variational functional rather than a pointwise PDE residual. This may be more relevant than a generic PINN for mechanics problems, but any future version should be rebuilt as a curve/energy-aware supervised loss or field-level model, not revived from the old notebook. **For predicting displacement/stress fields directly from geometry, Deep Ritz-style ideas might be more mechanically natural than a generic PINN**.

## 3. Generative Augmentation

Goal: generate plausible additional disorder-response examples or latent perturbations to improve robustness.

Preferred future routes:
- Use the existing Autoencoder as a starting point for latent-space augmentation.
- Consider conditional VAE-style generation before GANs.
- Use generative models to propose candidate inputs, then validate with FEA or active learning.
- Avoid using generated stress-strain curves as ground truth unless physically verified.

Why useful:
- Could help sparse regions of design/disorder space.
- Could support optimization and active sampling.
- Might expose structure in the learned latent representation.

Caution:
The old GAN notebook is not a good implementation basis. Any generative method should be rebuilt around the current DATA, MODEL, scaling/reconstruction, and diagnostics pipeline.